In [3]:
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from tqdm import tqdm

# ROOT_DIR = r"E:\NIT Delhi\Research 2\DDRseg"


In [4]:
class DDRSegDataset(Dataset):
    def __init__(self, root_dir, split='train', transform=None):
        self.img_dir = os.path.join(root_dir, split, 'image')
        self.label_dir = os.path.join(root_dir, split, 'label')
        self.lesion_types = ['MA', 'SE', 'EX', 'HE']
        self.filenames = [f for f in os.listdir(self.img_dir) if f.endswith('.jpg')]
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        # 1. Load Image
        img_name = self.filenames[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # 2. Merge 4 Binary Masks into 1 Multi-class Mask
        # BG=0, MA=1, SE=2, EX=3, HE=4
        mask_h, mask_w = image.shape[:2]
        merged_mask = np.zeros((mask_h, mask_w), dtype=np.uint8)

        for i, lesion in enumerate(self.lesion_types):
            # DDR binary masks are usually .tif files
            mask_path = os.path.join(self.label_dir, lesion, img_name.replace('.jpg', '.tif'))
            if os.path.exists(mask_path):
                binary_mask = cv2.imread(mask_path, 0)
                # Assign class value (1, 2, 3, or 4) to positive pixels
                merged_mask[binary_mask > 0] = i + 1

        if self.transform:
            augmented = self.transform(image=image, mask=merged_mask)
            image = augmented['image']
            mask = augmented['mask']

        return image, mask.long()

In [4]:
def get_model(num_classes=5):
    model = smp.Unet(
        encoder_name="resnet34",      # Reliable baseline
        encoder_weights="imagenet",   # Pre-trained on ImageNet
        in_channels=3,
        classes=num_classes,
        decoder_attention_type="scse" # This enables Attention Gates
    )
    return model

In [7]:
class EarlyStopping:
    def __init__(self, patience=7, delta=0):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.delta = delta

    def __call__(self, val_dice):
        if self.best_score is None:
            self.best_score = val_dice
        elif val_dice < self.best_score + self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_dice
            self.counter = 0

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for images, masks in tqdm(loader, desc="Training"):
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        loss.backward()
        optimizer.step()

        # --- CRITICAL: Gradient Clipping ---   newly added
        # Prevents gradients from exploding and causing 0.000 metrics
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        total_loss += loss.item()
    return total_loss / len(loader)

def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0
    dice_fn = smp.losses.DiceLoss(mode='multiclass')
    total_dice = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            
            val_loss += criterion(outputs, masks).item()
            total_dice += 1 - dice_fn(outputs, masks).item()
            
    return val_loss / len(loader), total_dice / len(loader)

In [8]:
def get_detailed_metrics(loader, model, device, num_classes=5):
    model.eval()
    names = ['Background', 'MA', 'SE', 'EX', 'HE']
    all_tp, all_fp, all_fn, all_tn = [], [], [], []

    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)

            tp, fp, fn, tn = smp.metrics.get_stats(
                preds, masks, mode='multiclass', num_classes=num_classes
            )
            all_tp.append(tp); all_fp.append(fp); all_fn.append(fn); all_tn.append(tn)

    tp = torch.cat(all_tp); fp = torch.cat(all_fp); fn = torch.cat(all_fn); tn = torch.cat(all_tn)

    # reduction="none" => returns tensor of shape (B, C) or (N, C) depending on stats stacking
    per_sample_iou = smp.metrics.iou_score(tp, fp, fn, tn, reduction="none")   # (..., C)
    per_sample_dice = smp.metrics.f1_score(tp, fp, fn, tn, reduction="none")   # (..., C)

    # Average across samples -> per-class
    per_class_iou = per_sample_iou.mean(dim=0)
    per_class_dice = per_sample_dice.mean(dim=0)

    results = {}
    for i in range(num_classes):
        results[f'{names[i]}_Dice'] = per_class_dice[i].item()
        results[f'{names[i]}_IoU'] = per_class_iou[i].item()

    results['mDice'] = per_class_dice.mean().item()
    results['mIoU'] = per_class_iou.mean().item()
    return results

In [6]:
# Hyperparameters
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 150
BATCH_SIZE = 8
LR = 1e-4

# Transforms (Standardization is key to avoid Black Masks)
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.VerticalFlip(p=0.5), # Add this
    A.RandomRotate90(p=0.5), # Add this
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5), # Vital for small lesions
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Initialize
# train_ds = DDRSegDataset(root_dir=ROOT_DIR, split='train', transform=train_transform)
# val_ds = DDRSegDataset(root_dir=ROOT_DIR, split='valid', transform=train_transform)
# train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

model = get_model().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

# Hybrid Loss: Focal + Dice
dice_loss = smp.losses.DiceLoss(mode='multiclass')
focal_loss = smp.losses.FocalLoss(mode='multiclass', gamma=2.5)
# criterion = lambda y_pred, y_true: dice_loss(y_pred, y_true) + (0.5 * focal_loss(y_pred, y_true))
# criterion = lambda y_pred, y_true: dice_loss(y_pred, y_true) + (2.0 * focal_loss(y_pred, y_true))
criterion = lambda y_pred, y_true: dice_loss(y_pred, y_true) + focal_loss(y_pred, y_true)


# early_stopping = EarlyStopping(patience=10)
# best_dice = 0

# for epoch in range(EPOCHS):
#     train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
#     val_loss, val_dice = validate(model, val_loader, criterion, DEVICE)
    
#     print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Val Dice: {val_dice:.4f}")
    
#     # Save Last Model
#     torch.save(model.state_dict(), "last_model.pth")
    
#     # Save Best Model
#     if val_dice > best_dice:
#         best_dice = val_dice
#         torch.save(model.state_dict(), "best_model.pth")
#         print("--> Saved Best Model")
        
#     early_stopping(val_dice)
#     if early_stopping.early_stop:
#         print("Early stopping triggered.")
#         break

In [6]:
# Initialize Scheduler
# This reduces the learning rate when the mDice plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

early_stopping = EarlyStopping(patience=20)
best_mdice = 0
num_classes = 5
lesion_names = ['MA', 'SE', 'EX', 'HE']

print(f"Starting Training on {DEVICE}...")

for epoch in range(EPOCHS):
    # 1. Training Phase
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    
    # 2. Validation Phase with Detailed Metrics
    val_metrics = get_detailed_metrics(val_loader, model, DEVICE, num_classes=num_classes)
    val_mdice = val_metrics['mDice']
    val_miou = val_metrics['mIoU']
    
    # 3. Print Detailed Logging
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    print(f"Train Loss: {train_loss:.4f} | Val mDice: {val_mdice:.4f} | Val mIoU: {val_miou:.4f}")
    
    # Print per-class breakdown to monitor for "Black Masks"
    for lesion in lesion_names:
        d_score = val_metrics[f'{lesion}_Dice']
        i_score = val_metrics[f'{lesion}_IoU']
        print(f" >> {lesion:2}: Dice = {d_score:.4f} | IoU = {i_score:.4f}")

    # 4. Save Last Model
    torch.save(model.state_dict(), "last_model.pth")
    
    # 5. Save Best Model based on mDice
    if val_mdice > best_mdice:
        best_mdice = val_mdice
        torch.save(model.state_dict(), "best_model.pth")
        print(f"*** New Best Model Saved (mDice: {best_mdice:.4f}) ***")
        
    # 6. Step the Scheduler
    scheduler.step(val_mdice)
        
    # 7. Early Stopping
    early_stopping(val_mdice)
    if early_stopping.early_stop:
        print("Early stopping triggered. Training terminated.")
        break

c:\Users\acer\.conda\envs\research\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Starting Training on cuda...


Training: 100%|██████████| 48/48 [01:46<00:00,  2.21s/it]



--- Epoch 1/150 ---
Train Loss: 1.4783 | Val mDice: 0.2061 | Val mIoU: 0.2024
 >> MA: Dice = 0.0006 | IoU = 0.0003
 >> SE: Dice = 0.0008 | IoU = 0.0004
 >> EX: Dice = 0.0004 | IoU = 0.0002
 >> HE: Dice = 0.0470 | IoU = 0.0470
*** New Best Model Saved (mDice: 0.2061) ***


Training: 100%|██████████| 48/48 [01:16<00:00,  1.59s/it]


...


--- Epoch 73/150 ---
Train Loss: 0.4049 | Val mDice: 0.6037 | Val mIoU: 0.5446
 >> MA: Dice = 0.2368 | IoU = 0.1593
 >> SE: Dice = 0.6872 | IoU = 0.6171
 >> EX: Dice = 0.5644 | IoU = 0.5110
 >> HE: Dice = 0.5314 | IoU = 0.4380


Training: 100%|██████████| 48/48 [01:17<00:00,  1.61s/it]



--- Epoch 74/150 ---
Train Loss: 0.4044 | Val mDice: 0.5998 | Val mIoU: 0.5408
 >> MA: Dice = 0.2395 | IoU = 0.1652
 >> SE: Dice = 0.7085 | IoU = 0.6347
 >> EX: Dice = 0.5591 | IoU = 0.5046
 >> HE: Dice = 0.4931 | IoU = 0.4022


Training: 100%|██████████| 48/48 [01:16<00:00,  1.59s/it]



--- Epoch 75/150 ---
Train Loss: 0.4029 | Val mDice: 0.5916 | Val mIoU: 0.5334
 >> MA: Dice = 0.2327 | IoU = 0.1591
 >> SE: Dice = 0.6892 | IoU = 0.6162
 >> EX: Dice = 0.5479 | IoU = 0.4970
 >> HE: Dice = 0.4896 | IoU = 0.3972
Early stopping triggered. Training terminated.


In [7]:
# def generate_test_masks(model_path, test_img_dir, output_root):
#     model = get_model().to(DEVICE)
#     model.load_state_dict(torch.load(model_path, map_location=DEVICE))
#     model.eval()

#     lesions = ['MA', 'SE', 'EX', 'HE']
#     for lesion in lesions:
#         os.makedirs(os.path.join(output_root, lesion), exist_ok=True)

#     # Only image files
#     exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
#     img_names = [f for f in os.listdir(test_img_dir) if f.lower().endswith(exts)]

#     for img_name in tqdm(img_names, desc="Generating test masks"):
#         img_path = os.path.join(test_img_dir, img_name)
#         img = cv2.imread(img_path)
#         if img is None:
#             print(f"Warning: could not read {img_path}, skipping.")
#             continue

#         orig_h, orig_w = img.shape[:2]

#         input_tensor = train_transform(image=cv2.cvtColor(img, cv2.COLOR_BGR2RGB))["image"]
#         input_tensor = input_tensor.unsqueeze(0).to(DEVICE)

#         with torch.no_grad():
#             pred = model(input_tensor).softmax(dim=1).argmax(dim=1).cpu().numpy()[0]  # 0..4

#         full_mask = cv2.resize(pred.astype(np.uint8), (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

#         for i, lesion in enumerate(lesions):
#             binary = (full_mask == (i + 1)).astype(np.uint8) * 255
#             out_name = os.path.splitext(img_name)[0] + ".tif"
#             cv2.imwrite(os.path.join(output_root, lesion, out_name), binary)

# # Correct path: test images folder
# generate_test_masks(
#     "best_model.pth",
#     os.path.join(ROOT_DIR, "test", "image"),
#     "results"
# )

In [9]:
def generate_test_masks(model_path, test_img_dir, test_label_dir, output_root):
    # 1. Setup Model
    model = get_model().to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    lesions = ['MA', 'SE', 'EX', 'HE']
    num_classes = 5 # BG + 4 lesions
    
    # Setup output folders
    for lesion in lesions:
        os.makedirs(os.path.join(output_root, lesion), exist_ok=True)

    # Filter image files
    exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
    img_names = [f for f in os.listdir(test_img_dir) if f.lower().endswith(exts)]
    
    # Metric accumulators
    all_tp, all_fp, all_fn, all_tn = [], [], [], []

    for img_name in tqdm(img_names, desc="Generating masks & calculating metrics"):
        # --- A. Load Image ---
        img_path = os.path.join(test_img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None: continue
        orig_h, orig_w = img.shape[:2]

        # --- B. Load Ground Truth (GT) for Metrics ---
        # Replicate the logic from DDRSegDataset to create a multi-class GT mask
        gt_mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
        for i, lesion in enumerate(lesions):
            # Assumes GT masks are .tif as per your dataset structure
            gt_path = os.path.join(test_label_dir, lesion, os.path.splitext(img_name)[0] + ".tif")
            if os.path.exists(gt_path):
                binary_gt = cv2.imread(gt_path, 0)
                gt_mask[binary_gt > 0] = i + 1

        # --- C. Predict ---
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        # Use the same transform used in validation
        input_tensor = train_transform(image=img_rgb)["image"].unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            output = model(input_tensor)
            pred = output.softmax(dim=1).argmax(dim=1).cpu().numpy()[0]  # Shape: (512, 512)

        # Resize prediction back to original image size for saving and metric calculation
        full_pred = cv2.resize(pred.astype(np.uint8), (orig_w, orig_h), interpolation=cv2.INTER_NEAREST)

        # --- D. Calculate Stats for this image ---
        # Convert to torch tensors for smp.metrics
        torch_pred = torch.from_numpy(full_pred).unsqueeze(0)
        torch_gt = torch.from_numpy(gt_mask).unsqueeze(0)
        
        tp, fp, fn, tn = smp.metrics.get_stats(
            torch_pred, torch_gt, mode='multiclass', num_classes=num_classes
        )
        all_tp.append(tp); all_fp.append(fp); all_fn.append(fn); all_tn.append(tn)

        # --- E. Save Binary Masks ---
        for i, lesion in enumerate(lesions):
            binary_out = (full_pred == (i + 1)).astype(np.uint8) * 255
            out_name = os.path.splitext(img_name)[0] + ".tif"
            cv2.imwrite(os.path.join(output_root, lesion, out_name), binary_out)

    # --- F. Aggregate and Print Final Test Metrics ---
    tp = torch.cat(all_tp); fp = torch.cat(all_fp); fn = torch.cat(all_fn); tn = torch.cat(all_tn)

    # per-image/per-class then average -> per-class
    per_sample_iou  = smp.metrics.iou_score(tp, fp, fn, tn, reduction="none", zero_division=0.0)
    per_sample_dice = smp.metrics.f1_score(tp, fp, fn, tn, reduction="none", zero_division=0.0)

    per_class_iou  = per_sample_iou.mean(dim=0)   # (C,)
    per_class_dice = per_sample_dice.mean(dim=0)  # (C,)

    print("\n" + "="*30)
    print("FINAL TEST METRICS SUMMARY")
    print("="*30)
    for i, lesion in enumerate(lesions):
        # Index i+1 because class 0 is background
        print(f"{lesion:3} | Dice: {per_class_dice[i+1]:.4f} | IoU: {per_class_iou[i+1]:.4f}")
    
    print("-" * 30)
    print(f"Mean Dice: {per_class_dice.mean():.4f}")
    print(f"Mean IoU:  {per_class_iou.mean():.4f}")
    print("="*30)

# --- Execution ---
generate_test_masks(
    model_path="best_model.pth",
    test_img_dir=os.path.join(ROOT_DIR, "test", "image"),
    test_label_dir=os.path.join(ROOT_DIR, "test", "label"), # Added label path
    output_root="results/test"
)
generate_test_masks(
    model_path="best_model.pth",
    test_img_dir=os.path.join(ROOT_DIR, "valid", "image"),
    test_label_dir=os.path.join(ROOT_DIR, "valid", "label"), # Added label path
    output_root="results/valid"
)

C:\Users\acer\AppData\Local\Temp\ipykernel_71292\1805065284.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=DEV


FINAL TEST METRICS SUMMARY
MA  | Dice: 0.0106 | IoU: 0.0075
SE  | Dice: 0.0084 | IoU: 0.0063
EX  | Dice: 0.0355 | IoU: 0.0240
HE  | Dice: 0.0229 | IoU: 0.0132
------------------------------
Mean Dice: 0.2142
Mean IoU:  0.2077


Generating masks & calculating metrics: 100%|██████████| 149/149 [01:33<00:00,  1.60it/s]


FINAL TEST METRICS SUMMARY
MA  | Dice: 0.0225 | IoU: 0.0149
SE  | Dice: 0.0430 | IoU: 0.0338
EX  | Dice: 0.0268 | IoU: 0.0195
HE  | Dice: 0.0360 | IoU: 0.0262
------------------------------
Mean Dice: 0.2251
Mean IoU:  0.2178


In [10]:
def generate_test_masks(model_path, test_img_dir, test_label_dir, output_root):
    # =========================
    # 1. Setup Model
    # =========================
    model = get_model().to(DEVICE)
    model.load_state_dict(torch.load(model_path, map_location=DEVICE))
    model.eval()

    lesions = ['MA', 'SE', 'EX', 'HE']
    num_classes = 5  # BG + 4 lesions
    IMG_SIZE = 512

    # Setup output folders
    for lesion in lesions:
        os.makedirs(os.path.join(output_root, lesion), exist_ok=True)

    # Filter image files
    exts = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp")
    img_names = [f for f in os.listdir(test_img_dir) if f.lower().endswith(exts)]

    # Metric accumulators
    all_tp, all_fp, all_fn, all_tn = [], [], [], []

    for img_name in tqdm(img_names, desc="Generating masks & calculating metrics"):
        # =========================
        # A. Load Original Image
        # =========================
        img_path = os.path.join(test_img_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            continue

        orig_h, orig_w = img.shape[:2]

        # =========================
        # B. Load Ground Truth (original size)
        # =========================
        gt_mask = np.zeros((orig_h, orig_w), dtype=np.uint8)

        for i, lesion in enumerate(lesions):
            gt_path = os.path.join(
                test_label_dir,
                lesion,
                os.path.splitext(img_name)[0] + ".tif"
            )
            if os.path.exists(gt_path):
                gt = cv2.imread(gt_path, cv2.IMREAD_UNCHANGED)
                if gt is None:
                    continue
                if gt.ndim == 3:
                    gt = gt[:, :, 0]
                gt = (gt > 0).astype(np.uint8)
                gt_mask[gt > 0] = i + 1

        # =========================
        # C. Resize Image → 512×512 (Inference)
        # =========================
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_512 = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)

        input_tensor = (
            train_transform(image=img_512)["image"]
            .unsqueeze(0)
            .to(DEVICE)
        )

        # =========================
        # D. Predict (512×512)
        # =========================
        with torch.no_grad():
            output = model(input_tensor)
            pred_512 = (
                output.softmax(dim=1)
                .argmax(dim=1)
                .cpu()
                .numpy()[0]
            )  # (512, 512)

        # =========================
        # E. Resize Prediction → Original Size
        # =========================
        full_pred = cv2.resize(
            pred_512.astype(np.uint8),
            (orig_w, orig_h),
            interpolation=cv2.INTER_NEAREST
        )

        # =========================
        # F. Metrics (Original Resolution)
        # =========================
        torch_pred = torch.from_numpy(full_pred).unsqueeze(0)
        torch_gt = torch.from_numpy(gt_mask).unsqueeze(0)

        tp, fp, fn, tn = smp.metrics.get_stats(
            torch_pred,
            torch_gt,
            mode='multiclass',
            num_classes=num_classes
        )

        all_tp.append(tp)
        all_fp.append(fp)
        all_fn.append(fn)
        all_tn.append(tn)

        # =========================
        # G. Save Binary Masks (Original Resolution)
        # =========================
        base_name = os.path.splitext(img_name)[0] + ".tif"

        for i, lesion in enumerate(lesions):
            binary_out = (full_pred == (i + 1)).astype(np.uint8) * 255
            cv2.imwrite(
                os.path.join(output_root, lesion, base_name),
                binary_out
            )

    # =========================
    # H. Aggregate Final Metrics
    # =========================
    tp = torch.cat(all_tp)
    fp = torch.cat(all_fp)
    fn = torch.cat(all_fn)
    tn = torch.cat(all_tn)

    per_sample_iou = smp.metrics.iou_score(
        tp, fp, fn, tn, reduction="none", zero_division=0.0
    )
    per_sample_dice = smp.metrics.f1_score(
        tp, fp, fn, tn, reduction="none", zero_division=0.0
    )

    per_class_iou = per_sample_iou.mean(dim=0)
    per_class_dice = per_sample_dice.mean(dim=0)

    print("\n" + "=" * 30)
    print("FINAL TEST METRICS SUMMARY")
    print("=" * 30)
    for i, lesion in enumerate(lesions):
        print(
            f"{lesion:3} | Dice: {per_class_dice[i+1]:.4f} "
            f"| IoU: {per_class_iou[i+1]:.4f}"
        )

    print("-" * 30)
    print(f"Mean Dice: {per_class_dice[1:].mean():.4f}")
    print(f"Mean IoU:  {per_class_iou[1:].mean():.4f}")
    print("=" * 30)


# --- Execution ---
generate_test_masks(
    model_path="best_model.pth",
    test_img_dir=os.path.join(ROOT_DIR, "test", "image"),
    test_label_dir=os.path.join(ROOT_DIR, "test", "label"), # Added label path
    output_root="results/test"
)
generate_test_masks(
    model_path="best_model.pth",
    test_img_dir=os.path.join(ROOT_DIR, "valid", "image"),
    test_label_dir=os.path.join(ROOT_DIR, "valid", "label"), # Added label path
    output_root="results/valid"
)

C:\Users\acer\AppData\Local\Temp\ipykernel_19424\2698637599.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=DEV


FINAL TEST METRICS SUMMARY
MA  | Dice: 0.0154 | IoU: 0.0095
SE  | Dice: 0.0102 | IoU: 0.0080
EX  | Dice: 0.0476 | IoU: 0.0323
HE  | Dice: 0.0406 | IoU: 0.0265
------------------------------
Mean Dice: 0.0284
Mean IoU:  0.0191


Generating masks & calculating metrics: 100%|██████████| 149/149 [05:02<00:00,  2.03s/it]


FINAL TEST METRICS SUMMARY
MA  | Dice: 0.0206 | IoU: 0.0123
SE  | Dice: 0.0461 | IoU: 0.0365
EX  | Dice: 0.0107 | IoU: 0.0074
HE  | Dice: 0.0351 | IoU: 0.0233
------------------------------
Mean Dice: 0.0281
Mean IoU:  0.0199


In [10]:
# Print model summary with all stages and outputs
# (Robust hooks for SMP models, print only unique shapes per stage)

def print_model_summary(model, input_size=(3, 512, 512)):
    import collections
    print("\nModel Summary (All Stages):\n" + "="*40)
    print(model)
    print("\n--- Layer-wise Output Shapes (using forward hooks) ---")
    hooks = []
    outputs = collections.OrderedDict()
    def hook_fn(module, input, output):
        name = module.__class__.__name__
        # Only print for tensor outputs
        if hasattr(output, 'shape'):
            shape = tuple(output.shape)
        elif isinstance(output, (list, tuple)):
            shape = [tuple(o.shape) if hasattr(o, 'shape') else type(o) for o in output]
        else:
            shape = type(output)
        # Only store first occurrence of each unique shape for each module type
        if name not in outputs:
            outputs[name] = shape
    # Register hooks for all leaf modules
    for module in model.modules():
        if len(list(module.children())) == 0:
            hooks.append(module.register_forward_hook(hook_fn))
    dummy = torch.randn(1, *input_size).to(next(model.parameters()).device)
    with torch.no_grad():
        _ = model(dummy)
    for k, v in outputs.items():
        print(f"{k}: {v}")
    for h in hooks:
        h.remove()

# Example usage:
print_model_summary(model)



Model Summary (All Stages):
Unet(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, mom

AttributeError: 'list' object has no attribute 'size'